In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the dataset

df = pd.read_csv("../Data/Raw/ChargingRecords.csv")

# Create a separate working DataFrame
df_cleaned = df.copy()

print("Original shape:", df.shape)
print("Working DataFrame shape:", df_cleaned.shape)

In [ ]:
# Remove Duplicates

df_cleaned = df_cleaned.drop_duplicates().copy()

print("Original records:", len(df))
print("Cleaned records:", len(df_cleaned))
print("Remaining duplicates:", df_cleaned.duplicated().sum())

In [ ]:
# Date columns
date_columns = ["StartDay", "EndDay"]

for column in date_columns:
    df_cleaned[column] = pd.to_datetime(
        df_cleaned[column],
        format="%Y-%m-%d"
    )

# Time columns
time_columns = ["StartTime", "EndTime"]

for column in time_columns:
    df_cleaned[column] = pd.to_datetime(
        df_cleaned[column],
        format="mixed"
    ).dt.time

# Datetime columns
datetime_columns = ["StartDatetime", "EndDatetime"]

for column in datetime_columns:
    df_cleaned[column] = pd.to_datetime(
        df_cleaned[column],
        format="mixed"
    )

# Verify data types
df_cleaned.info()

In [ ]:
# Create a separate dataset
df_duration_cleaned = df_cleaned.copy()

# Identify negative duration records
negative_mask = df_duration_cleaned["Duration"] < 0

# Swap start and end columns
columns_to_swap = [
    ("StartDatetime", "EndDatetime"),
    ("StartTime", "EndTime"),
    ("StartDay", "EndDay")
]

for start_col, end_col in columns_to_swap:
    df_duration_cleaned.loc[negative_mask, [start_col, end_col]] = (
        df_duration_cleaned.loc[negative_mask, [end_col, start_col]].to_numpy()
    )

# Recalculate corrected duration
df_duration_cleaned["CorrectedDuration"] = df_duration_cleaned["Duration"]

df_duration_cleaned.loc[negative_mask, "CorrectedDuration"] = (
    df_duration_cleaned.loc[negative_mask, "EndDatetime"]
    - df_duration_cleaned.loc[negative_mask, "StartDatetime"]
).dt.total_seconds() / 60

# Verify
print("Negative durations remaining:",
      (df_duration_cleaned["CorrectedDuration"] < 0).sum())

In [ ]:
display(
    df_duration_cleaned.loc[
        negative_mask,
        ["StartDatetime", "EndDatetime", "StartTime",
         "EndTime", "Duration", "CorrectedDuration"]
    ]
)

In [ ]:
# Reconstruct timestamps with second-level precision
start_exact = pd.to_datetime(
    df_duration_cleaned["StartDay"].dt.strftime("%Y-%m-%d")
    + " " + df_duration_cleaned["StartTime"].astype(str)
)

end_exact = pd.to_datetime(
    df_duration_cleaned["EndDay"].dt.strftime("%Y-%m-%d")
    + " " + df_duration_cleaned["EndTime"].astype(str)
)

# Calculate exact elapsed time in decimal minutes
elapsed_minutes = (
    end_exact - start_exact
).dt.total_seconds() / 60

In [ ]:
# Convert corrected duration to float
df_duration_cleaned["CorrectedDuration"] = (
    df_duration_cleaned["CorrectedDuration"].astype(float)
)

# Identify records that need duration correction
correction_mask = df_duration_cleaned["Duration"] <= 0

# Correct using exact elapsed minutes
valid_mask = correction_mask & (elapsed_minutes > 0)

df_duration_cleaned.loc[valid_mask, "CorrectedDuration"] = (
    elapsed_minutes.loc[valid_mask]
)

# Verify
print("Updated records:", valid_mask.sum())
print(
    "Remaining zero durations:",
    (df_duration_cleaned["CorrectedDuration"] == 0).sum()
)
print(
    "Negative corrected durations:",
    (df_duration_cleaned["CorrectedDuration"] < 0).sum()
)

In [ ]:
df_duration_cleaned[
    ["Duration", "CorrectedDuration"]
].describe()

In [ ]:
print("Negative Demand:", (df_duration_cleaned["Demand"] < 0).sum())
print("Zero Demand:", (df_duration_cleaned["Demand"] == 0).sum())
print("Missing Demand:", df_duration_cleaned["Demand"].isnull().sum())

In [ ]:
categorical_columns = [
    "Location",
    "ChargerType",
    "ChargerCompany"
]

for column in categorical_columns:
    print(f"\n{column}")
    print("Unique values:", df_duration_cleaned[column].nunique())
    print(df_duration_cleaned[column].value_counts())

In [ ]:
# Check whitespace
print(
    "Locations with leading/trailing spaces:",
    (df_duration_cleaned["Location"] !=
     df_duration_cleaned["Location"].str.strip()).sum()
)

# Check capitalization consistency
print(
    "Unique values after lowercase:",
    df_duration_cleaned["Location"].str.lower().nunique()
)

In [ ]:
print(
    "Duplicate records:",
    df_duration_cleaned.duplicated().sum()
)

In [ ]:
df_duration_cleaned.to_csv(
    "../Data/Cleaned/ChargingRecords_Cleaned.csv",
    index=False
)